# 새 외부 음원의 파일 진위 비교
T4 GPU에서 모두 실행하고 fresh_music_check_bundle.zip을 선택하세요.
실제 음악 후보 9개와 생성 음악 20개의 8초 구간을 원본/MP3 조건에서 비교합니다.
청취 미완료이므로 음악/음성 성분 지표나 대회 총점은 계산하지 않습니다. 학습·대회 제출 없음.
DF-Arena와 SONICS 가중치 다운로드가 필요할 수 있습니다. 결과: fresh_music_results.zip


In [ ]:
from google.colab import files
from pathlib import Path
import torch, sys, json, hashlib, zipfile, io, tempfile, shutil, subprocess, os
assert torch.cuda.is_available(), "런타임 유형을 T4 GPU로 바꾸세요."
print(torch.cuda.get_device_name(0), torch.__version__, sys.version)
uploaded=files.upload()
assert len(uploaded)==1, "fresh_music_check_bundle.zip 하나만 선택하세요."
blob=next(iter(uploaded.values()))
assert hashlib.sha256(blob).hexdigest()=="838dac0519d5339931b96d70f3927f6adc7ea011c8409c2247e56bc2bd1e5b59", "ZIP 버전이 다릅니다."
WORK=Path(tempfile.mkdtemp(prefix="fresh_music_check_",dir="/content"))
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    for name in z.namelist():
        assert (WORK/name).resolve().is_relative_to(WORK.resolve())
    assert z.testzip() is None
    z.extractall(WORK)
plan=json.loads((WORK/"plan.json").read_text())
for name,sha in plan["file_sha256"].items():
    assert hashlib.sha256((WORK/name).read_bytes()).hexdigest()==sha, name
RESULTS=WORK/"fresh_music_results"
RESULTS.mkdir()
print("29개 입력과 실행 코드 무결성 확인 완료")


In [ ]:
try:
    # torch/torchvision을 임의로 업그레이드하지 않는다.
    installed=subprocess.run([sys.executable,"-m","pip","install","--no-deps","timm==1.0.15"],capture_output=True,text=True)
    (RESULTS/"install.log").write_text(installed.stdout+installed.stderr)
    print((installed.stdout+installed.stderr)[-4000:])
    installed.check_returncode()
    extra=subprocess.run([sys.executable,"-m","pip","install","demucs==4.0.1"],capture_output=True,text=True)
    with (RESULTS/"install.log").open("a") as log:
        log.write(extra.stdout+extra.stderr)
    print((extra.stdout+extra.stderr)[-2000:])
    extra.check_returncode()
    check=subprocess.run([sys.executable,"-c","import timm, torch, torchvision, torchaudio, librosa, soundfile, huggingface_hub; from sonics import HFAudioClassifier; import script, transformers, sklearn; print('의존성 검사 통과', timm.__version__)"],cwd=WORK,capture_output=True,text=True)
    (RESULTS/"preflight.log").write_text(check.stdout+check.stderr)
    print(check.stdout+check.stderr)
    check.check_returncode()
    assert shutil.which("ffmpeg"), "ffmpeg가 없습니다."
    (RESULTS/"environment.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True)+"\n"+sys.version)

except Exception:
    import traceback
    (RESULTS/"setup_error.txt").write_text(traceback.format_exc())
    files.download(shutil.make_archive("/content/fresh_music_results","zip",RESULTS))
    raise


In [ ]:
from huggingface_hub import hf_hub_download

def digest_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(4*1024**2),b''):
            h.update(block)
    return h.hexdigest()

try:
    entries=[(entry,WORK/'weights'/variant/'pytorch_model.bin',entry['weights_sha256'])
             for variant,entry in plan['models'].items()]
    entries.append((plan['df_model'],WORK/'model/df_arena_1b/pytorch_model.bin',plan['weight_sha256']['df_arena']))
    for entry,target,expected in entries:
        target.parent.mkdir(parents=True,exist_ok=True)
        if target.is_file() and digest_file(target)==expected:
            continue
        if target.is_symlink():
            target.unlink()
        cached=Path(hf_hub_download(repo_id=entry['repo'],filename='pytorch_model.bin',revision=entry['revision']))
        assert digest_file(cached)==expected
        shutil.copy2(cached.resolve(strict=True),target)
        assert digest_file(target)==expected
        print('가중치 확인 완료',entry['repo'])
except Exception:
    import traceback
    (RESULTS/'download_error.txt').write_text(traceback.format_exc())
    files.download(shutil.make_archive('/content/fresh_music_results','zip',RESULTS))
    raise


In [ ]:
# 추론은 로컬 가중치를 사용하며 HF 네트워크 다운로드를 금지한다.
env=os.environ.copy()
env["HF_HUB_OFFLINE"]="1"
env["TRANSFORMERS_OFFLINE"]="1"
try:
    with (RESULTS/"run.log").open("w",encoding="utf-8") as log:
        process=subprocess.Popen([sys.executable,"-u","run_fresh_music_check.py","--work",str(WORK)],cwd=WORK,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        try:
            for line in process.stdout:
                print(line,end="")
                log.write(line)
                log.flush()
            code=process.wait()
        finally:
            if process.poll() is None:
                process.terminate()
                process.wait()
    assert code==0, "검증이 중단됐습니다. 결과 ZIP과 오류 내용을 보내주세요."
    summary=json.loads((RESULTS/'summary.json').read_text())
    assert summary['completed'] and summary['predictions']==58
    print('완료: 파일 진위 비교. 성분별 성능/대회 총점이 아닙니다.')
finally:
    archive=shutil.make_archive("/content/fresh_music_results","zip",RESULTS)
    files.download(archive)
